# 01 — LGBM v1: Walk-Forward Directional Model

**Architecture:** LightGBM binary classifier, M1Y expanding-window walk-forward.
**Target:** `label` — 1 if next-bar close > current close.
**Signal:** Long if P(Up) > threshold; Short if P(Up) < threshold.
**Fees:** Spot taker for long exits; futures taker for short exits; maker for TP/entries.

Loads `BTCUSDT_1h_unified.parquet`. Outputs standard artifacts for meta-learning.


In [ ]:

import calendar, itertools, json, time, warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
mpl.rcParams.update({
    'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 10, 'axes.titlesize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})
ACCENT='#F7931A'; BLUE='#2962FF'; GREY='#9E9E9E'; RED='#EF5350'; GREEN='#26A69A'

# ── Paths ─────────────────────────────────────────────────────────────────────
def _repo_root():
    p = Path.cwd()
    while p != p.parent:
        if (p / 'pyproject.toml').exists(): return p
        p = p.parent
    raise RuntimeError('pyproject.toml not found')

REPO     = _repo_root()
ARTS_DIR = REPO / 'artifacts' / 'notebooks_v2' / '01_lgbm'
ARTS_DIR.mkdir(parents=True, exist_ok=True)

# ── WFO ───────────────────────────────────────────────────────────────────────
OOS_START        = pd.Timestamp('2024-01-01')
GRID_VAL_START   = pd.Timestamp('2022-01-01')
GRID_VAL_END     = pd.Timestamp('2023-12-31')
TRAIN_WINDOW_H   = 8760      # 1 year
STEP_SIZE        = 720       # monthly step
EMBARGO          = 12
VAL_FRAC         = 0.20

# ── Fee model ─────────────────────────────────────────────────────────────────
MAKER_FEE         = 0.0000
SPOT_TAKER_FEE    = 0.0005
FUTURES_TAKER_FEE = 0.0005
BUFFER            = 0.0005
SHORT_FUNDING_H   = 0.0000077

# ── Features (Boruta-locked from v12) ─────────────────────────────────────────
SELECTED_FEATURES = [
    'close_vs_true_vwap', 'stoch_k_14', 'ret_2h', 'rsi_divergence',
    'close_vs_sma_7', 'bear_streak', 'close_vs_s1', 'macd_hist_5_13',
    'hurst_24h', 'ad_z_48h', 'ret_3h',
]
LABEL_COL = 'label'

# ── Backtest grid ─────────────────────────────────────────────────────────────
TRADING_GRID = {
    'long_threshold':  [0.55, 0.58, 0.60, 0.63],
    'short_threshold': [0.30, 0.35, 0.40],
    'entry_atr_mult':  [0.3,  0.6,  1.0],
    'sl_atr_mult':     [1.5,  2.0,  2.5],
    'tp_atr_mult':     [2.0,  2.5,  3.0],
    'min_sl':          [0.010, 0.015],
    'min_hold':        [4,  8],
    'max_hold':        [24, 48],
    'cooldown':        [2,  3],
}
_keys   = list(TRADING_GRID)
_combos = list(itertools.product(*TRADING_GRID.values()))
print(f'Grid: {len(_combos):,} combos  |  Features: {len(SELECTED_FEATURES)}')
print(f'Artifacts → {ARTS_DIR}')


## 1 · Load data

In [ ]:

UNIFIED = REPO / 'data' / 'features' / 'BTCUSDT_1h_unified.parquet'
if not UNIFIED.exists():
    raise FileNotFoundError(f'{UNIFIED} not found — run 00_data_ingestion_v1.ipynb first.')

df = pd.read_parquet(UNIFIED)
df.index = df.index.tz_localize(None) if df.index.tz else df.index

miss = [f for f in SELECTED_FEATURES if f not in df.columns]
if miss: print(f'WARNING — missing features: {miss}')
else:    print(f'All {len(SELECTED_FEATURES)} features present.')

oos_mask = df.index >= OOS_START
oos_df   = df[oos_mask].copy()
print(f'Total bars : {len(df):,}  ({df.index.min().date()} → {df.index.max().date()})')
print(f'OOS bars   : {len(oos_df):,}  ({OOS_START.date()} → {oos_df.index[-1].date()})')
print(f'Label dist : {df[LABEL_COL].value_counts().to_dict()}')


## 2 · M1Y walk-forward training

In [ ]:

def run_m1y_wfo(df, verbose=True):
    params = dict(num_leaves=31, max_depth=6, learning_rate=0.05,
                  colsample_bytree=0.5, min_child_samples=50, subsample=0.7,
                  reg_alpha=0.1, reg_lambda=1.0, n_estimators=500,
                  objective='binary', metric='auc', verbose=-1, random_state=42)
    n = len(df); probs = np.full(n, np.nan); i = 0; steps = 0
    last_model = None
    while i < n:
        tr_end = i; tr_start = max(0, tr_end - TRAIN_WINDOW_H)
        if tr_start >= tr_end - 100: i += STEP_SIZE; continue
        sl = df.iloc[tr_start:tr_end]
        val_n = max(50, int(len(sl) * VAL_FRAC))
        X_tr = sl.iloc[:-val_n][SELECTED_FEATURES].fillna(0).values
        y_tr = sl.iloc[:-val_n][LABEL_COL].values
        X_va = sl.iloc[-val_n:][SELECTED_FEATURES].fillna(0).values
        y_va = sl.iloc[-val_n:][LABEL_COL].values
        if len(np.unique(y_tr)) < 2: i += STEP_SIZE; continue
        mdl = lgb.LGBMClassifier(**params)
        mdl.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oos_end = min(i + STEP_SIZE, n); oos_emb = min(i + EMBARGO, oos_end)
        X_oos = df.iloc[oos_emb:oos_end][SELECTED_FEATURES].fillna(0).values
        if len(X_oos): probs[oos_emb:oos_end] = mdl.predict_proba(X_oos)[:, 1]
        last_model = mdl; steps += 1
        if verbose and steps % 6 == 1:
            print(f'  Step {steps:>3}  train [{df.index[tr_start].date()} → '
                  f'{df.index[tr_end-1].date()}]  OOS [{df.index[oos_emb].date()} → '
                  f'{df.index[oos_end-1].date()}]')
        i += STEP_SIZE
    return pd.Series(probs, index=df.index, name='p_up'), last_model

t0 = time.time()
all_probs, last_model = run_m1y_wfo(df)
print(f'WFO done in {(time.time()-t0)/60:.1f} min')

gv_mask  = (df.index >= GRID_VAL_START) & (df.index <= GRID_VAL_END)
oos_probs = all_probs[oos_mask]
gv_probs  = all_probs[gv_mask]

valid_oos = ~np.isnan(oos_probs.values)
auc_oos = roc_auc_score(oos_df[LABEL_COL].values[valid_oos], oos_probs.values[valid_oos])
print(f'OOS AUC: {auc_oos:.4f}  ({OOS_START.date()} → {oos_df.index[-1].date()})')


## 3 · Grid search (2022–2023 validation window)

In [ ]:

def _run_backtest(probs_arr, close_arr, high_arr, low_arr, atr_arr,
        long_threshold, short_threshold, entry_atr_mult, sl_atr_mult, tp_atr_mult,
        min_sl, min_hold, max_hold, cooldown, with_fees=True):
    n=len(close_arr); eq=np.ones(n); cur=1.0; trades=[]
    in_pos=False; direction=None; entry_px=sl_px=tp_px=pos_eq=entry_fee=0.0
    hold_cnt=cd_cnt=0; funding=0.0; pending=None
    for i in range(n):
        lo=low_arr[i]; hi=high_arr[i]; px=close_arr[i]
        if in_pos:
            hold_cnt+=1
            if direction=='short': funding+=SHORT_FUNDING_H
            eq[i]=pos_eq*(px/entry_px if direction=='long' else 1+(entry_px-px)/entry_px)
            exited=False; exit_px=0.0; reason=''; exit_fee=0.0
            if hold_cnt>=min_hold:
                if direction=='long':
                    if lo<=sl_px: exit_px=sl_px;exited=True;reason='sl';exit_fee=SPOT_TAKER_FEE if with_fees else 0.
                    elif hi>=tp_px: exit_px=tp_px;exited=True;reason='tp';exit_fee=MAKER_FEE
                    elif hold_cnt>=max_hold: exit_px=px;exited=True;reason='timeout';exit_fee=SPOT_TAKER_FEE if with_fees else 0.
                else:
                    if hi>=sl_px: exit_px=sl_px;exited=True;reason='sl';exit_fee=FUTURES_TAKER_FEE if with_fees else 0.
                    elif lo<=tp_px: exit_px=tp_px;exited=True;reason='tp';exit_fee=MAKER_FEE
                    elif hold_cnt>=max_hold: exit_px=px;exited=True;reason='timeout';exit_fee=FUTURES_TAKER_FEE if with_fees else 0.
            if exited:
                gross=((exit_px-entry_px)/entry_px if direction=='long' else (entry_px-exit_px)/entry_px)
                net=gross-(entry_fee+exit_fee if with_fees else 0.)+funding
                cur=pos_eq*(1.+net); eq[i]=cur
                trades.append({'direction':direction,'reason':reason,'gross':gross,'net':net,'hold':hold_cnt})
                in_pos=False; cd_cnt=cooldown; funding=0.
        elif pending is not None:
            d,lim,p_sl,p_tp=pending
            if d=='long': filled=lo<=lim+BUFFER; ef=MAKER_FEE if (filled and with_fees) else (SPOT_TAKER_FEE if with_fees else 0.)
            else: filled=hi>=lim-BUFFER; ef=MAKER_FEE if (filled and with_fees) else (FUTURES_TAKER_FEE if with_fees else 0.)
            entry_px=lim if filled else px; sl_px=p_sl; tp_px=p_tp; entry_fee=ef
            direction=d; in_pos=True; pos_eq=cur; hold_cnt=0; funding=0.; pending=None; eq[i]=cur
        elif cd_cnt>0: cd_cnt-=1; eq[i]=cur
        elif not np.isnan(probs_arr[i]) and i+1<n:
            atr=max(atr_arr[i],min_sl)
            if probs_arr[i]>long_threshold:
                pending=('long',px*(1-entry_atr_mult*atr),px*(1-sl_atr_mult*atr),px*(1+tp_atr_mult*atr))
            elif probs_arr[i]<short_threshold:
                pending=('short',px*(1+entry_atr_mult*atr),px*(1+sl_atr_mult*atr),px*(1-tp_atr_mult*atr))
            eq[i]=cur
        else: eq[i]=cur
    if in_pos:
        gross=((px-entry_px)/entry_px if direction=='long' else (entry_px-px)/entry_px)
        taker=SPOT_TAKER_FEE if direction=='long' else FUTURES_TAKER_FEE
        net=gross-(entry_fee+(taker if with_fees else 0.))+funding; cur=pos_eq*(1.+net); eq[-1]=cur
    return eq, trades

def _sharpe(eq):
    r=np.diff(np.log(np.maximum(eq,1e-12))); return float(r.mean()/(r.std(ddof=1)+1e-12)*np.sqrt(24*365))
def _maxdd(eq):
    pk=np.maximum.accumulate(eq); return float(((eq-pk)/(pk+1e-12)).min())

gv_df = df[gv_mask]
rows = []
t0 = time.time()
for vals in _combos:
    p = dict(zip(_keys, vals))
    if p['short_threshold'] >= p['long_threshold'] or p['max_hold'] < p['min_hold']: continue
    eq, tr = _run_backtest(gv_probs.values, gv_df['close'].values,
                           gv_df['high'].values, gv_df['low'].values,
                           gv_df['atr_14_pct'].values, with_fees=True, **p)
    if len(tr) < 20: continue
    rows.append({**p, 'sharpe':_sharpe(eq), 'total_ret':float(eq[-1]-1),
                 'maxdd':_maxdd(eq), 'win_rate':float(np.mean([t['net']>0 for t in tr])),
                 'n_trades':len(tr)})

grid_df = pd.DataFrame(rows).sort_values('sharpe', ascending=False).reset_index(drop=True)
INT = {'min_hold','max_hold','cooldown'}
BEST = {k:(int(grid_df.iloc[0][k]) if k in INT else float(grid_df.iloc[0][k])) for k in _keys}
print(f'Grid done in {time.time()-t0:.0f}s  |  {len(grid_df):,} valid combos')
print(f'Best params:'); [print(f'  {k:<18}{v}') for k,v in BEST.items()]
print(grid_df.head(5).to_string())


## 4 · OOS backtest

In [ ]:

eq_fees,tdf_fees=_run_backtest(oos_probs.values,oos_df['close'].values,
    oos_df['high'].values,oos_df['low'].values,oos_df['atr_14_pct'].values,with_fees=True,**BEST)
eq_0fee,tdf_0fee=_run_backtest(oos_probs.values,oos_df['close'].values,
    oos_df['high'].values,oos_df['low'].values,oos_df['atr_14_pct'].values,with_fees=False,**BEST)
TF=pd.DataFrame(tdf_fees) if tdf_fees else pd.DataFrame(columns=['direction','reason','gross','net','hold'])
T0=pd.DataFrame(tdf_0fee) if tdf_0fee else pd.DataFrame(columns=['direction','reason','gross','net','hold'])

bh=(oos_df['close'].values/oos_df['close'].iloc[0]-1)*100

print(f'{"":22}{"Trades":>7}{"Win":>8}{"Return":>9}{"Sharpe":>8}{"MaxDD":>8}')
print('-'*62)
for lbl,eq,t in [('With fees',eq_fees,TF),('Zero-fee',eq_0fee,T0)]:
    wr=(t['net']>0).mean() if len(t) else 0
    print(f'{lbl:22}{len(t):>7}{wr:>8.1%}{eq[-1]-1:>+9.1%}{_sharpe(eq):>8.3f}{_maxdd(eq):>8.1%}')
print(f'BTC Buy & Hold:                              {bh[-1]:>+9.1f}%')


## 5 · Figures

In [ ]:

o_idx = oos_df.index

# Equity + drawdown
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(13,7),height_ratios=[3,1],sharex=True)
ax1.plot(o_idx,(eq_fees-1)*100,color=ACCENT,lw=1.5,label='LGBM (w/ fees)')
ax1.plot(o_idx,(eq_0fee-1)*100,color=BLUE,lw=1.0,ls='--',label='LGBM (0-fee)')
ax1.plot(o_idx,bh,color=GREY,lw=1.0,ls=':',label='BTC B&H')
ax1.axhline(0,color=GREY,lw=0.6,ls=':'); ax1.set_ylabel('Return (%)'); ax1.legend()
ax1.set_title(f'LGBM v1 — OOS {OOS_START.date()}→{o_idx[-1].date()} | '
    f'Sharpe={_sharpe(eq_fees):.2f}  Return={eq_fees[-1]-1:+.1%}  MaxDD={_maxdd(eq_fees):.1%}  '
    f'AUC={auc_oos:.3f}', fontweight='bold')
pk=np.maximum.accumulate(eq_fees); dd=(eq_fees-pk)/pk*100
ax2.fill_between(o_idx,dd,0,color=RED,alpha=0.4); ax2.set_ylabel('DD (%)')
ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
plt.setp(ax2.xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.tight_layout(); fig.savefig(ARTS_DIR/'01_equity_drawdown.png'); plt.show()


In [ ]:

# Monthly returns
import calendar
eqs=pd.Series(eq_fees,index=o_idx); mret=eqs.resample('ME').last().pct_change().fillna(0)*100
fig,ax=plt.subplots(figsize=(13,4))
ax.bar(mret.index,mret.values,color=[GREEN if r>=0 else RED for r in mret],width=22,alpha=0.75)
ax.plot(mret.index,mret.rolling(3,min_periods=1).mean(),color=BLUE,lw=2,label='3-mo SMA')
ax.axhline(0,color=GREY,lw=0.8,ls=':'); ax.set_ylabel('Return (%)'); ax.legend()
ax.set_title(f'Monthly Returns — LGBM v1 | positive {int((mret>0).sum())}/{len(mret)}'
             f' avg {mret.mean():+.2f}%', fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,4,7,10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right')
fig.tight_layout(); fig.savefig(ARTS_DIR/'02_monthly_returns.png'); plt.show()

cal=mret.to_frame('r'); cal['y']=cal.index.year; cal['m']=cal.index.month
piv=cal.pivot(index='y',columns='m',values='r'); piv.columns=[calendar.month_abbr[m] for m in piv.columns]
fig,ax=plt.subplots(figsize=(12,max(2,len(piv)*0.7)))
sns.heatmap(piv,ax=ax,cmap='RdYlGn',center=0,annot=True,fmt='.1f',linewidths=0.5)
ax.set_title('Monthly Return Calendar — LGBM v1',fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel(''); fig.tight_layout()
fig.savefig(ARTS_DIR/'03_monthly_heatmap.png'); plt.show()


## 6 · Save artifacts

In [ ]:

# ── Standard artifacts for meta-learning ─────────────────────────────────────
np.save(ARTS_DIR / 'oos_probs.npy',  oos_probs.values.astype(np.float32))
np.save(ARTS_DIR / 'oos_index.npy',  oos_df.index.astype(np.int64).values)

# Save last-fold model (LightGBM text format)
if last_model is not None:
    last_model.booster_.save_model(str(ARTS_DIR / 'model.txt'))
    print('Model saved → model.txt')

def _bt_metrics(eq, t):
    wr=float((t['net']>0).mean()) if len(t) else 0.
    nl=int((t['direction']=='long').sum()) if len(t) else 0
    ns=int((t['direction']=='short').sum()) if len(t) else 0
    return {'n_trades':len(t),'n_long':nl,'n_short':ns,'win_rate':round(wr,4),
            'total_ret':round(float(eq[-1]-1),4),'sharpe':round(_sharpe(eq),4),
            'maxdd':round(_maxdd(eq),4)}

results = {
    'notebook':'01_lgbm_v1','created':pd.Timestamp.now().isoformat(),
    'model':'LightGBM binary (directional label)',
    'wfo':{'train_window_h':TRAIN_WINDOW_H,'step_size':STEP_SIZE,'embargo':EMBARGO},
    'grid_val_window':f'{GRID_VAL_START.date()}→{GRID_VAL_END.date()}',
    'oos_period':f'{OOS_START.date()}→{oos_df.index[-1].date()}',
    'oos_auc':round(float(auc_oos),4),
    'selected_features':SELECTED_FEATURES,
    'best_params':BEST,
    'backtest_wfees':_bt_metrics(eq_fees,TF),
    'backtest_0fee':_bt_metrics(eq_0fee,T0),
    'monthly':{'mean_pct':round(float(mret.mean()),3),
               'positive_months':int((mret>0).sum()),'total_months':int(len(mret))},
    'artifacts':{'oos_probs':'oos_probs.npy','oos_index':'oos_index.npy','model':'model.txt'},
}
with open(ARTS_DIR/'results.json','w') as f: json.dump(results,f,indent=2)
print(f'Artifacts saved → {ARTS_DIR}')
print(json.dumps({k:v for k,v in results.items() if k!='selected_features'},indent=2)[:1200])
